In [15]:
import zipfile

input_3mf = "Task_1_Obj_1.gcode.3mf"
extracted_gcode = "extracted.gcode"
output_3mf = "modified_2.gcode.3mf"

with zipfile.ZipFile(input_3mf, "r") as src:
    # Find the G-code file inside the archive
    gcode_path = [f for f in src.namelist() if f.endswith(".gcode")][0]
    
    # Read and write out to a standalone file
    with open(extracted_gcode, "wb") as f_out:
        f_out.write(src.read(gcode_path))

print(f"Extracted internal file '{gcode_path}' to '{extracted_gcode}'")

Extracted internal file 'Metadata/plate_1.gcode' to 'extracted.gcode'


In [16]:
from gemini_pause_2 import add_pause_every_n_layers

output_path = add_pause_every_n_layers(
    input_gcode="extracted.gcode",
    every_n=5,
    absolute_z=100,
    dwell_ms=2000,
    output_gcode="modified_2.gcode",
)

print(output_path)

C:\Users\marco\volco\gcodes_with_pause\modified_2.gcode


In [17]:
with zipfile.ZipFile(input_3mf, "r") as src, zipfile.ZipFile(output_3mf, "w", zipfile.ZIP_DEFLATED) as dst:
    for item in src.infolist():
        if item.filename.endswith(".gcode"):
            # Replace internal G-code entry with your edited local file
            dst.write('modified_2.gcode', arcname=item.filename)
        else:
            # Preserve all original project settings and thumbnails
            dst.writestr(item, src.read(item.filename))

print(f"Reinserted modified G-code into {output_3mf}")

Reinserted modified G-code into modified_2.gcode.3mf


In [23]:
from io import BytesIO
import time
import zipfile
import bambulabs_api as bl
import os

IP = '172.16.97.160'
SERIAL = '01P00C641402808'
ACCESS_CODE = '42117904'

# ============================================================
INPUT_FILE_PATH = "modified_2.gcode.3mf"
UPLOAD_FILE_NAME = "modified_2.3mf"
PLATE_ID = 1  # Target plate inside the 3MF file (usually 1)
# ============================================================

env = os.getenv("env", "debug")
plate = os.getenv("plate", "true").lower() == "true"


def create_zip_archive_in_memory(
        text_content: str,
        text_file_name: str = 'file.txt') -> BytesIO:
    """
    Create a zip archive in memory

    Args:
        text_content (str): content of the text file
        text_file_name (str, optional): location of the text file.
            Defaults to 'file.txt'.

    Returns:
        io.BytesIO: zip archive in memory
    """
    zip_buffer = BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.writestr(text_file_name, text_content)
    zip_buffer.seek(0)
    return zip_buffer


if __name__ == '__main__':
    print('Starting bambulabs_api example')
    print('Connecting to Bambulabs 3D printer')
    print(f'IP: {IP}')
    print(f'Serial: {SERIAL}')
    print(f'Access Code: {ACCESS_CODE}')

    # Create a new instance of the API
    printer = bl.Printer(IP, ACCESS_CODE, SERIAL)

    # Connect to the Bambulabs 3D printer
    printer.connect()

    time.sleep(5)

    with open(INPUT_FILE_PATH, "r") as file:
        gcode = file.read()

    if plate:
        gcode_location = "Metadata/plate_1.gcode"
        io_file = create_zip_archive_in_memory(gcode, gcode_location)
        if file:
            result = printer.upload_file(io_file, UPLOAD_FILE_NAME)
            if "226" not in result:
                print("Error Uploading File to Printer")

            else:
                print("Done Uploading/Sending Start Print Command")
                #printer.start_print(UPLOAD_FILE_NAME, 1)
                #print("Start Print Command Sent")
    else:
        gcode_location = INPUT_FILE_PATH
        io_file = create_zip_archive_in_memory(gcode, gcode_location)
        if file:
            result = printer.upload_file(io_file, UPLOAD_FILE_NAME)
            if "226" not in result:
                print("Error Uploading File to Printer")

            else:
                print("Done Uploading/Sending Start Print Command")
                #printer.start_print(UPLOAD_FILE_NAME, gcode_location)
                #print("Start Print Command Sent")

Starting bambulabs_api example
Connecting to Bambulabs 3D printer
IP: 172.16.97.160
Serial: 01P00C641402808
Access Code: 42117904


UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 70: character maps to <undefined>

Error occurred: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond
Error occurred: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond
Error occurred: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond
Error occurred: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond
Error occurred: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time

In [25]:
import os
import time
import bambulabs_api as bl

# Configuration
IP = os.getenv("PRINTER_IP", '172.16.97.160')
SERIAL = os.getenv("PRINTER_SERIAL", '01P00C641402808')
ACCESS_CODE = os.getenv("PRINTER_ACCESS_CODE", '42117904')

INPUT_FILE_PATH = "modified_2.gcode.3mf"
UPLOAD_FILE_NAME = "modified_2.3mf"
PLATE_ID = 1  # Target plate inside the 3MF file (usually 1)

if __name__ == "__main__":
    print(f"Connecting to printer at {IP}...")
    printer = bl.Printer(IP, ACCESS_CODE, SERIAL)
    printer.connect()
    time.sleep(5)

    # Read and upload the existing .gcode.3mf file directly as binary
    with open(INPUT_FILE_PATH, "rb") as file:
        result = printer.upload_file(file, UPLOAD_FILE_NAME)

    if "226" in str(result):
        print("Upload successful")
        #printer.start_print(UPLOAD_FILE_NAME, PLATE_ID)
        #print("Start Print command sent.")
    else:
        print(f"Error Uploading File to Printer. Response: {result}")

Connecting to printer at 172.16.97.160...
Upload successful. Starting print...


Error occurred: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond
Error occurred: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond
